# 개별종목 조합F — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합F 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합F의 피처 값만 지정합니다.
import json

COMBINATION = 'F'
FEATURE_COLUMNS = (
    'ret_5_rank',
    'sector_relative_rank',
    'turnover_rank',
    'hv_20_rank',
    'market_cap_percentile',
    'sector_market_cap_rank',
    'industry_stock_rank',
    'volume_z_20',
    'bb_position',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합F 피처: ('ret_5_rank', 'sector_relative_rank', 'turnover_rank', 'hv_20_rank', 'market_cap_percentile', 'sector_market_cap_rank', 'industry_stock_rank', 'volume_z_20', 'bb_position')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.4407,0.5012,-0.0605,0.3720,0.3031,0.3633
1,2,balanced,980,20150123,20150421,0.3734,0.3978,-0.0244,0.3615,0.3081,0.3452
2,3,balanced,1210,20151228,20160328,0.3594,0.3762,-0.0168,0.3513,0.3223,0.3436
3,4,balanced,1439,20161202,20170228,0.4182,0.4617,-0.0435,0.4030,0.3868,0.4023
4,5,balanced,1669,20171113,20180207,0.3875,0.3901,-0.0026,0.3780,0.3464,0.3698
5,6,balanced,1899,20181024,20190118,0.4092,0.3725,0.0367,0.4001,0.3284,0.3756
6,7,balanced,2129,20190930,20191224,0.4071,0.4781,-0.0710,0.3881,0.4032,0.3993
7,8,balanced,2359,20200902,20201130,0.3912,0.3476,0.0435,0.3879,0.3713,0.3832
8,9,balanced,2589,20210806,20211105,0.3909,0.3916,-0.0007,0.3832,0.3237,0.3633
9,10,balanced,2818,20220714,20221012,0.3492,0.3454,0.0038,0.3398,0.3016,0.3288


,OOS 폴드 평균
accuracy,0.3923
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0045
macro_f1,0.3775
down_recall,0.3394
core_harmonic_mean,0.3678


재실행 명령: python scripts/run_stock_model_experiment.py
